In [ ]:
# ==========================================
# Import Libraries
# ==========================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries Imported Successfully")

In [ ]:
DATA_PATH = "../data/processed/"

df = pd.read_csv(DATA_PATH + "feature_engineered.csv")

print(df.shape)

df.head()

In [ ]:
df.columns.tolist()

In [ ]:
if "Revenue" not in df.columns:

    if "price" in df.columns and "quantity" in df.columns:

        df["Revenue"] = df["price"] * df["quantity"]

In [ ]:
abc = (

    df.groupby("sku_id")

    ["Revenue"]

    .sum()

    .reset_index()

)

In [ ]:
abc = abc.sort_values(

    by="Revenue",

    ascending=False

)

In [ ]:
abc["Revenue_Percentage"] = (

    abc["Revenue"]

    /

    abc["Revenue"].sum()

) * 100

abc["Cumulative"] = abc["Revenue_Percentage"].cumsum()

abc.head()

In [ ]:
def abc_class(value):

    if value <= 80:

        return "A"

    elif value <= 95:

        return "B"

    else:

        return "C"


abc["ABC_Class"] = abc["Cumulative"].apply(abc_class)

abc.head()

In [ ]:
plt.figure(figsize=(10,5))

sns.countplot(

data=abc,

x="ABC_Class"

)

plt.title("ABC Analysis")

plt.show()

In [ ]:
xyz = (

df.groupby("sku_id")

["quantity"]

.mean()

.reset_index()

)

xyz.rename(

columns={

"quantity":"Average_Demand"

},

inplace=True
)

In [ ]:
variation = (

df.groupby("sku_id")

["quantity"]

.std()

.reset_index()

)

variation.rename(

columns={

"quantity":"Demand_STD"

},

inplace=True
)

xyz = xyz.merge(

variation,

on="sku_id"

)

In [ ]:
xyz["CV"] = (

xyz["Demand_STD"]

/

xyz["Average_Demand"]

)

xyz.head()

In [ ]:
def xyz_class(cv):

    if cv <= 0.5:

        return "X"

    elif cv <= 1:

        return "Y"

    else:

        return "Z"

xyz["XYZ_Class"] = xyz["CV"].apply(xyz_class)

In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(

data=xyz,

x="XYZ_Class"

)

plt.show()

In [ ]:
inventory = (

df.groupby("sku_id")

["quantity"]

.sum()

.reset_index()

)

inventory.rename(

columns={

"quantity":"Units_Sold"

},

inplace=True)

inventory.head()

In [ ]:
stock = (

df.groupby("sku_id")

["stock_quantity"]

.mean()

.reset_index()

)

inventory = inventory.merge(

stock,

on="sku_id"

)

In [ ]:
inventory["Turnover"] = (

inventory["Units_Sold"]

/

(inventory["stock_quantity"]+1)

)

In [ ]:
inventory["Stock_Out_Risk"] = np.where(

inventory["stock_quantity"]<20,

"High",

"Low"

)

In [ ]:
inventory["Overstock"] = np.where(

inventory["stock_quantity"]>

inventory["Units_Sold"]*2,

"Yes",

"No"

)

In [ ]:
inventory["Safety_Stock"] = (

1.65 *

xyz["Demand_STD"]

)

In [ ]:
Lead_Time = 7

inventory["Reorder_Point"] = (

inventory["Safety_Stock"]

+

Lead_Time *

xyz["Average_Demand"]

)

In [ ]:
Annual_Demand = inventory["Units_Sold"]

Ordering_Cost = 100

Holding_Cost = 20

inventory["EOQ"] = np.sqrt(

(

2*

Annual_Demand*

Ordering_Cost

)

/

Holding_Cost

)

In [ ]:
def recommendation(row):

    if row["Stock_Out_Risk"]=="High":

        return "Reorder Immediately"

    elif row["Overstock"]=="Yes":

        return "Reduce Purchase"

    else:

        return "Inventory Healthy"

inventory["Recommendation"] = inventory.apply(

recommendation,

axis=1

)

In [ ]:
inventory = inventory.merge(

abc[["sku_id","ABC_Class"]],

on="sku_id"

)

In [ ]:
inventory = inventory.merge(

xyz[["sku_id","XYZ_Class"]],

on="sku_id"

)

In [ ]:
inventory.head()

In [ ]:
plt.figure(figsize=(10,5))

sns.countplot(

data=inventory,

x="Recommendation"

)

plt.xticks(rotation=20)

plt.show()

In [ ]:
critical = inventory[

inventory["Recommendation"]

=="Reorder Immediately"

]

critical.head(10)

In [ ]:
inventory.to_csv(

DATA_PATH+

"inventory_recommendation.csv",

index=False

)

print("Inventory Report Saved")

In [ ]:
summary = {

"Total Products":len(inventory),

"ABC A Products":

len(

inventory[

inventory["ABC_Class"]=="A"

]

),

"High Risk":

len(

inventory[

inventory["Stock_Out_Risk"]=="High"

]

),

"Overstock":

len(

inventory[

inventory["Overstock"]=="Yes"

]

)

}

summary

In [ ]:
print("="*60)

print("PHASE 6 COMPLETED")

print("="*60)

print("""
✔ ABC Analysis Completed
✔ XYZ Analysis Completed
✔ Inventory Turnover Calculated
✔ Stock-Out Risk Calculated
✔ Overstock Detection Completed
✔ Safety Stock Calculated
✔ Reorder Point Calculated
✔ EOQ Calculated
✔ AI Recommendations Generated
✔ Ready for Dashboard
""")